# Phase 4b: Training & Evaluation

### Fixes vs previous version:
1. **No data leakage**: train with `gnn_features_train.pkl`, eval with `gnn_features_test.pkl`
2. **Actual budget_ratio/step_ratio** in pretrain (not hardcoded)
3. **Reward shaping** implemented (MINERVA-style potential-based)
4. **GreedyGNN baseline** evaluated for comparison
5. **Both approaches** trained: Traversal RL + Path Selection RL

### Training strategy:
- **Stage 1**: Imitation Learning (Behavior Cloning from GNN-RAG paths)
- **Stage 2**: REINFORCE with baseline + reward shaping

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/GraphRAG_Research/utils')
from common import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import pickle
import json
from tqdm import tqdm
from collections import defaultdict
from sentence_transformers import SentenceTransformer

In [ ]:
# === Copy classes from P4a (GNNHeuristic, Embedder, policies, retrievers, etc.) ===
# TODO: Copy all class definitions from P4a cells here
print('Copy classes from P4a notebook here before running')

## 1. Load Data (NO LEAKAGE)

**CRITICAL**: Use TRAIN GNN features for training, TEST for evaluation only.

In [ ]:
# FIX: Separate train/test GNN features
gnn_train = GNNHeuristic(os.path.join(CHECKPOINT_DIR, 'gnn_features_train.pkl'))
gnn_test = GNNHeuristic(os.path.join(CHECKPOINT_DIR, 'gnn_features_test.pkl'))

print(f'Train GNN features: {len(gnn_train.features)} questions')
print(f'Test GNN features: {len(gnn_test.features)} questions')

if not gnn_train.features:
    print('\nWARNING: No train features! Run P2 cell "6b" to generate train.info first.')

# Embedder
EMB_CACHE = os.path.join(CHECKPOINT_DIR, 'embeddings_cache.pkl')
embedder = Embedder(cache_path=EMB_CACHE)
EMB_DIM = embedder.emb_dim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2. Evaluate GreedyGNN Baseline (no training needed)

In [ ]:
def evaluate_retriever(retriever, gnn, max_samples=None):
    """Evaluate any retriever. Check entity overlap with gold answers."""
    results = []
    qids = list(gnn.features.keys())
    if max_samples:
        qids = qids[:max_samples]
    
    for qid in tqdm(qids, desc='Evaluating'):
        qdata = gnn.get(qid)
        gold = set(gnn.get_answers(qid))
        
        r = retriever.retrieve(qid) if not hasattr(retriever, 'policy') else retriever.retrieve(qid, greedy=True)
        
        collected = set()
        for h, rel, t in r['triples']:
            collected.add(str(h))
            collected.add(str(t))
        
        overlap = collected & gold
        hits = 1.0 if overlap else 0.0
        recall = len(overlap) / max(len(gold), 1)
        precision = len(overlap) / max(len(collected), 1)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        results.append({'qid': qid, 'f1': f1, 'hits': hits, 'recall': recall,
                        'cost': r['cost'], 'n_triples': len(r['triples'])})
    
    summary = {
        'n': len(results),
        'f1': np.mean([r['f1'] for r in results]),
        'hits': np.mean([r['hits'] for r in results]),
        'recall': np.mean([r['recall'] for r in results]),
        'cost': np.mean([r['cost'] for r in results]),
    }
    print(f'  N={summary["n"]} | F1={summary["f1"]:.4f} | Hits={summary["hits"]:.4f} | '
          f'Recall={summary["recall"]:.4f} | Cost={summary["cost"]:.1f}')
    return summary, results


# === Greedy GNN Baseline (no training) ===
print('=== GreedyGNN Baseline (test set) ===')
greedy_retriever = GreedyGNNRetriever(gnn_test, max_steps=5, budget=50)
greedy_summary, greedy_results = evaluate_retriever(greedy_retriever, gnn_test, max_samples=500)

with open(os.path.join(RESULTS_DIR, 'greedy_gnn_webqsp.json'), 'w') as f:
    json.dump({'summary': greedy_summary}, f, indent=2)
print('Saved greedy baseline results.')

## 3. Stage 1: Imitation Learning (FIX: actual budget_ratio)

In [ ]:
def extract_expert_trajectories(gnn, max_samples=None):
    """Extract expert trajectories from TRAIN GNN features."""
    trajectories = []
    qids = list(gnn.features.keys())
    if max_samples:
        qids = qids[:max_samples]
    
    for qid in tqdm(qids, desc='Extracting expert trajectories'):
        qdata = gnn.get(qid)
        G = gnn.get_local_graph(qid)
        if G.number_of_edges() == 0:
            continue
        
        seeds = gnn.get_seeds(qid)
        candidates = qdata.get('top_candidates', [])[:10]
        if not seeds or not candidates:
            continue
        
        for seed in seeds:
            if seed not in G:
                continue
            for cand in candidates:
                if cand not in G or cand == seed:
                    continue
                try:
                    path = nx.shortest_path(G, seed, cand)
                except nx.NetworkXNoPath:
                    continue
                if len(path) < 2:
                    continue
                
                actions = []
                for i in range(len(path) - 1):
                    cur, nxt = path[i], path[i + 1]
                    neighbors = list(G.neighbors(cur))
                    if nxt in neighbors:
                        rel = G[cur][nxt].get('relation', 'related_to')
                        actions.append({
                            'current': cur, 'next': nxt, 'relation': rel,
                            'neighbors': neighbors,
                            'step_in_path': i,
                            'total_steps': len(path) - 1,
                        })
                
                if actions:
                    trajectories.append({'qid': qid, 'question': qdata['question'],
                                        'actions': actions})
    
    print(f'Extracted {len(trajectories)} trajectories')
    return trajectories


# FIX: Use TRAIN data
expert_trajs = extract_expert_trajectories(gnn_train)

In [ ]:
def pretrain_traversal(policy, embedder, expert_trajs, gnn,
                        num_epochs=10, lr=1e-3, budget=50,
                        max_steps=10, device='cpu'):
    """IL pre-training with ACTUAL budget_ratio and relation encoding."""
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    policy.train()
    
    for epoch in range(num_epochs):
        total_loss = 0
        correct = 0
        total = 0
        np.random.shuffle(expert_trajs)
        
        for traj in expert_trajs:
            qid = traj['qid']
            query_emb = embedder.encode(traj['question']).to(device)
            visited = [traj['actions'][0]['current']]
            
            for step in traj['actions']:
                cur = step['current']
                target = step['next']
                neighbors = step['neighbors']
                if len(neighbors) <= 1 or target not in neighbors:
                    continue
                
                # Build action list: neighbors + STOP
                act_ent = []
                act_rel = []
                gnn_scores = []
                G = gnn.get_local_graph(qid)
                
                for nb in neighbors:
                    act_ent.append(embedder.encode(nb))
                    rel = G[cur][nb].get('relation', 'related_to') if G.has_edge(cur, nb) else 'related_to'
                    act_rel.append(embedder.encode(rel))
                    gnn_scores.append(gnn.get_node_score(qid, nb))
                
                # STOP action
                act_ent.append(policy.stop_entity_emb.detach())
                act_rel.append(policy.stop_relation_emb.detach())
                gnn_scores.append(0.0)
                
                act_ent_t = torch.stack(act_ent).unsqueeze(0).to(device)
                act_rel_t = torch.stack(act_rel).unsqueeze(0).to(device)
                gnn_t = torch.tensor(gnn_scores).unsqueeze(0).unsqueeze(-1).float().to(device)
                
                node_emb = embedder.encode(cur).unsqueeze(0).to(device)
                hist = embedder.encode_batch(visited).unsqueeze(0).to(device)
                
                # FIX: actual budget/step ratios
                b_ratio = torch.tensor([[max(0, (budget - len(visited)) / budget)]]).to(device)
                s_ratio = torch.tensor([[step['step_in_path'] / max_steps]]).to(device)
                
                probs = policy(query_emb.unsqueeze(0), node_emb, hist,
                              b_ratio, s_ratio, act_ent_t, act_rel_t, gnn_t)
                
                expert_idx = neighbors.index(target)
                loss = F.nll_loss(torch.log(probs + 1e-8),
                                 torch.tensor([expert_idx], device=device))
                
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
                optimizer.step()
                
                total_loss += loss.item()
                correct += (probs.argmax().item() == expert_idx)
                total += 1
                visited.append(target)
        
        avg_loss = total_loss / max(total, 1)
        acc = correct / max(total, 1)
        print(f'Epoch {epoch+1}/{num_epochs}: Loss={avg_loss:.4f}, Acc={acc:.4f}')
        
        torch.save({'epoch': epoch+1, 'model_state': policy.state_dict(),
                    'loss': avg_loss, 'acc': acc},
                   os.path.join(CHECKPOINT_DIR, f'traversal_pretrain_e{epoch+1}.pt'))
    
    return policy


print('pretrain_traversal defined (with actual budget_ratio + relation encoding).')

In [ ]:
# === Run pre-training ===
# policy_traversal = TraversalPolicy(emb_dim=EMB_DIM).to(device)
# policy_traversal = pretrain_traversal(
#     policy_traversal, embedder, expert_trajs, gnn_train,
#     num_epochs=10, lr=1e-3, device=device
# )

print('Pre-training (uncomment when ready)')

## 4. Stage 2: RL Fine-tuning (with reward shaping)

In [ ]:
class RLTrainer:
    """REINFORCE with baseline + reward shaping (MINERVA-style)."""
    
    def __init__(self, retriever, gnn, embedder,
                 lr=1e-4, gamma=0.99, use_shaping=True,
                 reward_weights=(0.6, 0.2, 0.2), device='cpu'):
        self.retriever = retriever
        self.gnn = gnn
        self.embedder = embedder
        self.gamma = gamma
        self.use_shaping = use_shaping
        self.alpha, self.beta, self.gamma_w = reward_weights
        self.device = device
        
        self.optimizer = torch.optim.Adam(retriever.policy.parameters(), lr=lr)
        self.baseline = 0.0
        self.baseline_decay = 0.99
        self.log = {'rewards': [], 'losses': [], 'costs': []}
    
    def _reward_shaping_potential(self, qid, entities):
        """Potential function phi(s) = max GNN score among entities.
        Used for MINERVA-style reward shaping.
        """
        if not entities:
            return 0.0
        scores = [self.gnn.get_node_score(qid, e) for e in entities]
        return max(scores) if scores else 0.0
    
    def train_episode(self, qid):
        self.retriever.policy.train()
        qdata = self.gnn.get(qid)
        if qdata is None:
            return 0.0, 0.0, 0
        
        result = self.retriever.retrieve(qid, greedy=False)
        if not result['log_probs']:
            return 0.0, 0.0, result['cost']
        
        gold = qdata.get('answers', [])
        terminal_reward = compute_reward(
            result['triples'], gold,
            result['cost'], self.retriever.budget
            if hasattr(self.retriever, 'budget') else self.retriever.max_paths,
            alpha=self.alpha, beta=self.beta, gamma=self.gamma_w
        )
        
        # Reward shaping (MINERVA-style)
        reward = terminal_reward
        if self.use_shaping and hasattr(self.retriever, 'env'):
            # Add potential-based shaping: gamma*phi(s') - phi(s)
            env = self.retriever.env
            path = env.path
            shaping_bonus = 0.0
            for i in range(1, len(path)):
                phi_prev = self._reward_shaping_potential(qid, path[:i])
                phi_curr = self._reward_shaping_potential(qid, path[:i+1])
                shaping_bonus += self.gamma * phi_curr - phi_prev
            reward += 0.1 * shaping_bonus  # Scale down shaping
        
        # REINFORCE
        advantage = reward - self.baseline
        self.baseline = self.baseline_decay * self.baseline + (1 - self.baseline_decay) * reward
        
        log_probs = torch.stack(result['log_probs'])
        loss = -(log_probs.sum() * advantage)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.retriever.policy.parameters(), 1.0)
        self.optimizer.step()
        
        self.log['rewards'].append(reward)
        self.log['losses'].append(loss.item())
        self.log['costs'].append(result['cost'])
        
        return loss.item(), reward, result['cost']
    
    def train(self, num_epochs=5, log_every=100):
        qids = list(self.gnn.features.keys())
        
        for epoch in range(num_epochs):
            np.random.shuffle(qids)
            epoch_r, epoch_l = [], []
            
            for i, qid in enumerate(tqdm(qids, desc=f'Epoch {epoch+1}')):
                loss, reward, cost = self.train_episode(qid)
                epoch_r.append(reward)
                epoch_l.append(loss)
                
                if (i + 1) % log_every == 0:
                    print(f'  Step {i+1}: R={np.mean(epoch_r[-log_every:]):.4f}')
            
            print(f'Epoch {epoch+1}: R={np.mean(epoch_r):.4f}')
            torch.save({
                'epoch': epoch+1,
                'model_state': self.retriever.policy.state_dict(),
                'baseline': self.baseline,
            }, os.path.join(CHECKPOINT_DIR, f'rl_epoch{epoch+1}.pt'))
        
        return self.log


print('RLTrainer defined (with reward shaping).')

In [ ]:
# === Train Traversal RL ===
# policy_traversal = TraversalPolicy(emb_dim=EMB_DIM).to(device)
# # Load pretrain checkpoint first:
# # ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'traversal_pretrain_e10.pt'))
# # policy_traversal.load_state_dict(ckpt['model_state'])
# 
# traversal_retriever = TraversalRetriever(
#     gnn_train, policy_traversal, embedder,
#     max_steps=10, budget=50, device=device
# )
# trainer = RLTrainer(traversal_retriever, gnn_train, embedder,
#                     lr=1e-4, use_shaping=True, device=device)
# trav_log = trainer.train(num_epochs=5)

print('Traversal RL training (uncomment when ready)')

In [ ]:
# === Train Path Selection RL ===
# policy_selection = PathSelectionPolicy(emb_dim=EMB_DIM).to(device)
# selection_retriever = PathSelectionRetriever(
#     gnn_train, policy_selection, embedder,
#     max_paths=10, device=device
# )
# trainer_sel = RLTrainer(selection_retriever, gnn_train, embedder,
#                         lr=1e-4, use_shaping=False, device=device)
# sel_log = trainer_sel.train(num_epochs=5)

print('Selection RL training (uncomment when ready)')

## 5. Final Evaluation (on TEST set)

In [ ]:
# === Evaluate all methods on TEST set ===
# IMPORTANT: Use gnn_test (not gnn_train)

# print('\n=== 1. GreedyGNN Baseline ===')
# greedy = GreedyGNNRetriever(gnn_test, max_steps=5, budget=50)
# greedy_s, _ = evaluate_retriever(greedy, gnn_test)
# 
# print('\n=== 2. Traversal RL ===')
# trav = TraversalRetriever(gnn_test, policy_traversal, embedder,
#                           max_steps=10, budget=50, device=device)
# trav_s, _ = evaluate_retriever(trav, gnn_test)
# 
# print('\n=== 3. Path Selection RL ===')
# sel = PathSelectionRetriever(gnn_test, policy_selection, embedder,
#                              max_paths=10, device=device)
# sel_s, _ = evaluate_retriever(sel, gnn_test)
# 
# print('\n=== Comparison ===')
# import pandas as pd
# df = pd.DataFrame([greedy_s, trav_s, sel_s],
#                    index=['GreedyGNN', 'Traversal RL', 'Selection RL'])
# print(df[['f1', 'hits', 'recall', 'cost']].to_markdown())

print('Final evaluation (uncomment after training)')

## 6. Checkpoint Management

In [ ]:
def load_checkpoint(policy, path):
    ckpt = torch.load(path, map_location='cpu')
    policy.load_state_dict(ckpt['model_state'])
    print(f'Loaded: {path} (epoch {ckpt.get("epoch", "?")})')
    return ckpt.get('baseline', 0.0)


def find_latest(prefix, ckpt_dir=None):
    import glob
    d = ckpt_dir or CHECKPOINT_DIR
    files = glob.glob(os.path.join(d, f'{prefix}*.pt'))
    return max(files, key=os.path.getmtime) if files else None


print('Checkpoint utils defined.')

## Summary

### Fixes applied:
| Issue | Fix |
|-------|-----|
| Data leakage | `gnn_train.pkl` for training, `gnn_test.pkl` for eval |
| Hardcoded budget_ratio=0.8 | Actual `(budget - visited) / budget` |
| No reward shaping | MINERVA-style potential-based shaping |
| No baseline comparison | GreedyGNN evaluated first |
| Relation ignored in actions | Relation embedding added |
| Budget per seed | Shared budget across seeds |

### Results to compare in P5:
1. GreedyGNN (no RL, no training)
2. GRL-RAG Traversal (IL + RL on graph)
3. GRL-RAG Selection (RL selects paths)
4. GNN-RAG (published results)